# **Load & Examine Data**

In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

intake = pd.read_csv('AAC_Intakes.csv')     # load Intake df
outcome = pd.read_csv('AAC_Outcomes.csv')   # load Outcome df

In [2]:

# view column names and determine data types (intake)
intake.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173812 entries, 0 to 173811
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Animal ID         173812 non-null  object
 1   Name              123821 non-null  object
 2   DateTime          173812 non-null  object
 3   MonthYear         173812 non-null  object
 4   Found Location    173812 non-null  object
 5   Intake Type       173812 non-null  object
 6   Intake Condition  173812 non-null  object
 7   Animal Type       173812 non-null  object
 8   Sex upon Intake   173811 non-null  object
 9   Age upon Intake   173812 non-null  object
 10  Breed             173812 non-null  object
 11  Color             173812 non-null  object
dtypes: object(12)
memory usage: 15.9+ MB


In [3]:

# view column names and determine data types (outcome)
outcome.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173775 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Animal ID         173775 non-null  object
 1   Date of Birth     173775 non-null  object
 2   Name              123991 non-null  object
 3   DateTime          173775 non-null  object
 4   MonthYear         173775 non-null  object
 5   Outcome Type      173729 non-null  object
 6   Outcome Subtype   79660 non-null   object
 7   Animal Type       173775 non-null  object
 8   Sex upon Outcome  173774 non-null  object
 9   Age upon Outcome  173766 non-null  object
 10  Breed             173775 non-null  object
 11  Color             173775 non-null  object
dtypes: object(12)
memory usage: 15.9+ MB


## View the first 5 rows of each table

In [4]:

# intake
intake.head()

,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
0,A521520,Nina,10/01/2013 07:51:00 AM,October 2013,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,7 years,Border Terrier/Border Collie,White/Tan
1,A664235,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
2,A664236,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
3,A664237,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
4,A664233,Stevie,10/01/2013 08:53:00 AM,October 2013,7405 Springtime in Austin (TX),Stray,Injured,Dog,Intact Female,3 years,Pit Bull Mix,Blue/White


In [5]:

# outcome
outcome.head()

,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green
3,A679066,2014-04-16,NaN,2014-05-16T00:00:00-05:00,05-2014,NaN,NaN,Other,Unknown,4 weeks,Rabbit Sh,Brown
4,A680855,2014-05-25,NaN,2014-06-10T00:00:00-05:00,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black


-----
# **Pre-processing**

## Narrow outcomes: Adoption only
removing:
 - 'Euthanasia', 'Disposal', and 'Transfer'
   *(These quick exits would be treated as a 'win', and they don't reflect the goal of the project)*
 - 'Return To Owner' and 'RTO - Adopt'
   *(These represent reunions, not adoptions. Including these could cause target bias)*

In [6]:
print(f"\nBefore: {outcome.shape[0]:,} animals") # animals before filtering

# keeping only successful adoptions
outcome = outcome[outcome['Outcome Type'] == 'Adoption']

print(f" After: {outcome.shape[0]:,} animals") # animals after filtering


Before: 173,775 animals
 After: 84,598 animals


## ~~Found Location~~

In [7]:
# looking only at the first 5 rows of the 'Found Location' column, I've already found manual entry errors, such as "Norht" instead of "North".
# I've decided to drop the 'Found Location' column.
# this feature creates excessive noise. Standardizing it would require too much pre-processing for a feature with such high cardinality.
intake = intake.drop(columns=['Found Location'])

-----
# **Standardize Data**

## DateTime

In [8]:
# standardize and rename intake date and time 
intake['i_datetime'] = pd.to_datetime(intake['DateTime'])  

# standardize and rename outcome date and time
outcome['o_datetime'] = pd.to_datetime(
    outcome['DateTime'], 
    format = 'ISO8601', 
    utc = True
).dt.tz_localize(None)

# check that both columns have been standardized
print('\ni_datetime type:', intake['i_datetime'].dtype)
print('o_datetime type:', outcome['o_datetime'].dtype)

# drop the original DateTime columns from both tables
intake = intake.drop(columns=['DateTime'])
outcome = outcome.drop(columns=['DateTime'])


i_datetime type: datetime64[ns]
o_datetime type: datetime64[ns]


## Intake: MonthYear: String -> Int

In [9]:
# create Int variables for intake month and intake year, extracted from intake DateTime
intake['i_month'] = intake['i_datetime'].dt.month
intake['i_year'] = intake['i_datetime'].dt.year

# drop MonthYear (String) from intake table
intake = intake.drop(columns=['MonthYear'])

## Age at Intake: String -> Int

In [10]:
# check for animals without an age value
print(intake['Age upon Intake'].isna().sum())

0


In [11]:
# function: convert age as a string to an integer (== age in days)
def age_to_days(age_string):

    # split the string into 2 substrings
    parts = str(age_string).split()
    number = abs(int(parts[0])) # taking abs value accounts for the 10 negative values in the dataset. it's assumed that the negative (-) was unintended
    unit = parts[1].lower() 

    # convert to days
    if 'year' in unit: return number * 365
    elif 'month' in unit: return number * 30
    elif 'week' in unit: return number * 7
    else: return number   # if already in days

# apply function to intake df and create column with new name  
intake['i_age_days'] = intake['Age upon Intake'].apply(age_to_days)

# drop original age column
intake = intake.drop(columns=['Age upon Intake'])

# check values for age as int
print(intake['i_age_days'])

0         2555
1            7
2            7
3            7
4         1095
          ... 
173807     730
173808     365
173809     365
173810    3650
173811     730
Name: i_age_days, Length: 173812, dtype: int64


## Species: Filter to Cats and Dogs

In [12]:
print(intake['Animal Type'].value_counts(dropna=False)) # display all animal types processed through intake

Animal Type
Dog          94608
Cat          69324
Other         8968
Bird           878
Livestock       34
Name: count, dtype: int64


*By removing nondomestic species, the variance in the dataset is reduced and keeps focus on the center's primary mission: domestic pet placement.*

In [13]:

# remove any row where type != 'cat' or 'dog'
intake = intake[intake['Animal Type'].isin(['Dog', 'Cat'])] 

# create dummy variables
intake = pd.get_dummies(intake, columns=['Animal Type'], dtype=int)

# drop 'Dog' because as it's the majority, it will function as the baseline for this feature
intake = intake.drop(columns=['Animal Type_Dog'])

# rename the remaining feature to 'is_cat' (1 = cat, 0 = dog)
intake = intake.rename(columns={'Animal Type_Cat': 'is_cat'})

# check feature names
intake.columns.tolist()

['Animal ID',
 'Name',
 'Intake Type',
 'Intake Condition',
 'Sex upon Intake',
 'Breed',
 'Color',
 'i_datetime',
 'i_month',
 'i_year',
 'i_age_days',
 'is_cat']

## Reproductive Status: (M/F) (Fixed/Intact)

In [14]:
# display values and frequencies for 'Sex upon Intake'
print(intake['Sex upon Intake'].value_counts())

Sex upon Intake
Intact Male      57910
Intact Female    55827
Neutered Male    24148
Spayed Female    20262
Unknown           5784
Name: count, dtype: int64


In [15]:

# determine if female. if yes, return 1
def is_female(sex):    
    if pd.isna(sex): return 0 # keep from crashing if NaN
    return 1 if 'Female' in str(sex) else 0

# determine if fixed. if yes, return 1
def is_fixed(sex):
    if pd.isna(sex): return 0 # keep from crashing
    return 1 if 'Spayed' in str(sex) or 'Neutered' in str(sex) else 0

# determine if null or 'Unknown'
def is_unknown_sex(sex):
    if pd.isna(sex) or 'Unknown' in str(sex): return 1
    return 0

# apply the features to the table
intake['sex_female'] = intake['Sex upon Intake'].apply(is_female)
intake['sex_unknown'] = intake['Sex upon Intake'].apply(is_unknown_sex)
intake['status_fixed'] = intake['Sex upon Intake'].apply(is_fixed)

# drop the original feature
intake = intake.drop(columns=['Sex upon Intake'])

# check
intake.info()


<class 'pandas.core.frame.DataFrame'>
Index: 163932 entries, 0 to 173811
Data columns (total 14 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Animal ID         163932 non-null  object        
 1   Name              121971 non-null  object        
 2   Intake Type       163932 non-null  object        
 3   Intake Condition  163932 non-null  object        
 4   Breed             163932 non-null  object        
 5   Color             163932 non-null  object        
 6   i_datetime        163932 non-null  datetime64[ns]
 7   i_month           163932 non-null  int32         
 8   i_year            163932 non-null  int32         
 9   i_age_days        163932 non-null  int64         
 10  is_cat            163932 non-null  int64         
 11  sex_female        163932 non-null  int64         
 12  sex_unknown       163932 non-null  int64         
 13  status_fixed      163932 non-null  int64         
dtypes: dateti

## Name: Does the animal have one?

In [16]:
# check whether animals are likely to have a name at intake by viewing frequency of each
print(intake['Name'].isnull().map({False: 'Has Name', True: 'No Name'}).value_counts().map('{:,}'.format))

Name
Has Name    121,971
No Name      41,961
Name: count, dtype: object


In [17]:

# because the majority of animals have a name at intake:
# if 'Name' is missing or is 'Unknown', return 1
def check_no_name(name):
    if pd.isnull(name) or str(name).lower() == 'unknown':
        return 1
    return 0   # 'has name' is baseline

# create the feature
intake['no_name'] = intake['Name'].apply(check_no_name)

# drop the original 'Name' feature
intake = intake.drop(columns=['Name'])

# check # of unnamed animals
print(f"Unnamed at intake: {intake['no_name'].sum():,} of {len(intake):,}")

Unnamed at intake: 42,013 of 163,932


## Animal's Condition

In [18]:
# check the frequency of each condition for 1% threshold
print(intake['Intake Condition'].value_counts()) # frequency of each
print("-----------------------------")
# check the frequency of each value
print(intake['Intake Condition'].value_counts(normalize=True)*100) # as a percentage


Intake Condition
Normal        140805
Injured         9339
Sick            6043
Nursing         3773
Neonatal        1943
Medical          604
Aged             522
Other            339
Pregnant         170
Feral            144
Med Attn          87
Behavior          81
Unknown           29
Med Urgent        21
Neurologic        11
Parvo             11
Space              4
Agonal             4
Panleuk            1
Congenital         1
Name: count, dtype: int64
-----------------------------
Intake Condition
Normal        85.892321
Injured        5.696874
Sick           3.686285
Nursing        2.301564
Neonatal       1.185248
Medical        0.368445
Aged           0.318425
Other          0.206793
Pregnant       0.103702
Feral          0.087841
Med Attn       0.053071
Behavior       0.049411
Unknown        0.017690
Med Urgent     0.012810
Neurologic     0.006710
Parvo          0.006710
Space          0.002440
Agonal         0.002440
Panleuk        0.000610
Congenital     0.000610
Name: prop

In [19]:
# create features for conditions with frequencies of >1,639 (1%) 
def check_condition(condition):
    top_cond = ['Normal', 'Injured', 'Sick', 'Nursing', 'Neonatal']
    if condition in top_cond:
        return condition.lower() # return condition name in lowercase
    return 'other' # any condition <1% will go into 'other'

# apply function to intake df
intake['cond_temp'] = intake['Intake Condition'].apply(check_condition)

# create dummy variables
cond_dummies = pd.get_dummies(intake['cond_temp'], prefix='cond', dtype=int)

# add condition features to df 
intake = pd.concat([intake, cond_dummies], axis=1)

# drop original 'Intake Condition' column, baseline feature 'cond_normal', and temporary feature 'cond_temp' 
intake = intake.drop(columns=['Intake Condition', 'cond_normal', 'cond_temp'])



# check that the column names are correctly updated
intake.columns.tolist()

['Animal ID',
 'Intake Type',
 'Breed',
 'Color',
 'i_datetime',
 'i_month',
 'i_year',
 'i_age_days',
 'is_cat',
 'sex_female',
 'sex_unknown',
 'status_fixed',
 'no_name',
 'cond_injured',
 'cond_neonatal',
 'cond_nursing',
 'cond_other',
 'cond_sick']

## Intake Type

In [20]:
# check the frequency of each intake type for the 1% threshold
print(intake['Intake Type'].value_counts()) # frequency of each
print("-----------------------------")
print(intake['Intake Type'].value_counts(normalize=True)*100) # as a percentage

Intake Type
Stray                 117484
Owner Surrender        34475
Public Assist           9866
Abandoned               1863
Euthanasia Request       243
Wildlife                   1
Name: count, dtype: int64
-----------------------------
Intake Type
Stray                 71.666301
Owner Surrender       21.030061
Public Assist          6.018349
Abandoned              1.136447
Euthanasia Request     0.148232
Wildlife               0.000610
Name: proportion, dtype: float64


In [21]:

# function: categorize intake types
def check_type(intake_type):
    top_types = ['Stray', 'Owner Surrender', 'Public Assist', 'Abandoned'] # significant (>=1%) categories

    if intake_type in top_types:
        return intake_type.lower().replace(' ', '_') # return as lowercase with underscores replacing spaces

    return 'other'   # <1% 

# apply function to intake df, creating a temporary column
intake['type_temp'] = intake['Intake Type'].apply(check_type)

# create dummiy variables
type_dummies = pd.get_dummies(intake['type_temp'], prefix='i_type', dtype=int)

# add new features to df and manually drop baseline ('stray')
intake = pd.concat([intake, type_dummies], axis=1)
# drop original column 'Intake Type', temporary column 'Temp_Type', and intake type baseline 'i_type_stray'
intake = intake.drop(columns=['Intake Type', 'type_temp', 'i_type_stray'])



# check that the column names are correctly updated
intake.columns.tolist()

['Animal ID',
 'Breed',
 'Color',
 'i_datetime',
 'i_month',
 'i_year',
 'i_age_days',
 'is_cat',
 'sex_female',
 'sex_unknown',
 'status_fixed',
 'no_name',
 'cond_injured',
 'cond_neonatal',
 'cond_nursing',
 'cond_other',
 'cond_sick',
 'i_type_abandoned',
 'i_type_other',
 'i_type_owner_surrender',
 'i_type_public_assist']

## **Cats**

### Breeds

In [22]:
# look at top 20 cat breeds (frequency as %)
print((intake[intake['is_cat'] == 1]['Breed'].value_counts(normalize=True)*100).head(20))

Breed
Domestic Shorthair Mix        48.561826
Domestic Shorthair            34.889793
Domestic Medium Hair Mix       4.809301
Domestic Medium Hair           3.053776
Domestic Longhair Mix          2.420518
Siamese Mix                    2.121920
Domestic Longhair              1.151116
Siamese                        0.787606
Snowshoe Mix                   0.343315
American Shorthair Mix         0.323120
Maine Coon Mix                 0.178870
Manx Mix                       0.160118
Russian Blue Mix               0.128383
American Shorthair             0.089435
Ragdoll Mix                    0.075010
Russian Blue                   0.073568
Himalayan Mix                  0.062028
Snowshoe                       0.060585
Maine Coon                     0.054815
Siamese/Domestic Shorthair     0.033178
Name: proportion, dtype: float64


In [23]:
print('\nNumber of unique breeds in df: ', intake[intake['is_cat'] == 1]['Breed'].nunique())


Number of unique breeds in df:  111


In [24]:

# convert mixed breed types such as 'Domestic Shorthair/Siamese' to 'Domestic Shorthair Mix'
def clean_mixes(original):
    if '/' in str(original):
        primary = original.split('/')[0].strip()  # split into substrings at the '/' and keep the first substring
        return primary + ' Mix' # append ' Mix' to the string
    return original

# apply 'cleaned cat breeds' to df
intake.loc[intake['is_cat'] == 1, 'Breed'] = intake.loc[intake['is_cat'] == 1, 'Breed'].apply(clean_mixes)

# check
print((intake[intake['is_cat'] == 1]['Breed'].value_counts(normalize=True)*100).head(15))

Breed
Domestic Shorthair Mix      48.586348
Domestic Shorthair          34.889793
Domestic Medium Hair Mix     4.819399
Domestic Medium Hair         3.053776
Domestic Longhair Mix        2.437828
Siamese Mix                  2.165195
Domestic Longhair            1.151116
Siamese                      0.787606
Snowshoe Mix                 0.350528
American Shorthair Mix       0.323120
Maine Coon Mix               0.178870
Manx Mix                     0.175985
Russian Blue Mix             0.128383
American Shorthair           0.089435
Ragdoll Mix                  0.080780
Name: proportion, dtype: float64


In [25]:
print('\nNumber of unique breeds after cleaning: ', intake[intake['is_cat'] == 1]['Breed'].nunique())


Number of unique breeds after cleaning:  72


In [26]:

# list of significant (>=1%) cat breeds
top_cat_breeds = ['Domestic Shorthair Mix', 'Domestic Shorthair', 'Domestic Medium Hair Mix', 'Domestic Medium Hair', 'Domestic Longhair Mix', 'Siamese Mix', 'Domestic Longhair']

def check_cat_breed(cat):
    if cat['is_cat'] == 1:                    # if cat:
        if cat['Breed'] in top_cat_breeds:    # if breed is in the list:
            return cat['Breed'].lower()       # return the name in lowercase
        else:
            return 'other'        # if not in list: goes to 'other' bucket
    return None                   # if not a cat: skip it.

# apply the function to create a temp column
intake['Temp_catBreed'] = intake.apply(check_cat_breed, axis=1)
# clean the breed string for use as dummy columns (replacing ' ' with '_')
intake['Temp_catBreed'] = intake['Temp_catBreed'].str.replace(' ', '_')

# create the dummy columns
dummies = pd.get_dummies(intake['Temp_catBreed'], prefix='c_breed', dtype=int)
# add dummy columns to table
intake = pd.concat([intake, dummies], axis=1)

# drop temp column and baseline ('Domestic Shorthair Mix')
intake = intake.drop(columns=['Temp_catBreed', 'c_breed_domestic_shorthair_mix'])


# check that the column names are correctly updated
intake.columns.tolist()


['Animal ID',
 'Breed',
 'Color',
 'i_datetime',
 'i_month',
 'i_year',
 'i_age_days',
 'is_cat',
 'sex_female',
 'sex_unknown',
 'status_fixed',
 'no_name',
 'cond_injured',
 'cond_neonatal',
 'cond_nursing',
 'cond_other',
 'cond_sick',
 'i_type_abandoned',
 'i_type_other',
 'i_type_owner_surrender',
 'i_type_public_assist',
 'c_breed_domestic_longhair',
 'c_breed_domestic_longhair_mix',
 'c_breed_domestic_medium_hair',
 'c_breed_domestic_medium_hair_mix',
 'c_breed_domestic_shorthair',
 'c_breed_other',
 'c_breed_siamese_mix']

### Colors

In [27]:
# look at top cat colors (%)
print((intake[intake['is_cat'] == 1]['Color'].value_counts(normalize=True)*100).head(18))

Color
Brown Tabby           15.469390
Black                 12.945012
Black/White            8.989672
Brown Tabby/White      7.865963
Orange Tabby           7.423115
Tortie                 4.561191
Calico                 4.284231
Blue Tabby             4.063528
Blue                   3.692805
Orange Tabby/White     3.688477
Torbie                 3.105707
Blue/White             2.577751
Cream Tabby            1.886792
Blue Tabby/White       1.844960
White/Black            1.686285
Lynx Point             1.309792
White/Brown Tabby      1.058796
White                  0.934741
Name: proportion, dtype: float64


In [28]:
# create list of significant (>=1%) cat colors
top_cat_colors = ['Brown Tabby', 'Black', 'Black/White', 'Brown Tabby/White', 'Orange Tabby', 'Tortie', 'Calico', 'Blue Tabby', 'Blue', 'Orange Tabby/White', 'Torbie',
                 'Blue/White', 'Cream Tabby', 'Blue Tabby/White', 'White/Black', 'Lynx Point', 'White/Brown Tabby']

def check_cat_color(cat):
    if cat['is_cat'] == 1:                       # if cat:
        if cat['Color'] in top_cat_colors:       # if the color in the list:
            return cat['Color'].lower()          # keep the name in lowercase
        else:
            return 'other'       # if color not in list: goes to 'other' bucket
    return None                  # if not a cat: skip it.

# apply the function to create a temp column
intake['Temp_Color'] = intake.apply(check_cat_color, axis=1)

# clean the color string for use as dummy columns (replacing ' ' with '_')
intake['Temp_Color'] = intake['Temp_Color'].str.replace(' ', '_').str.replace('/', '_')

# create the dummy columns
dummies = pd.get_dummies(intake['Temp_Color'], prefix='c_color', dtype=int)

# add dummy columns to table
intake = pd.concat([intake, dummies], axis=1)

# drop temp column and baseline color ('Brown Tabby')
intake = intake.drop(columns=['Temp_Color', 'c_color_brown_tabby'])



# check that the column names are correctly updated
intake.columns.tolist()

['Animal ID',
 'Breed',
 'Color',
 'i_datetime',
 'i_month',
 'i_year',
 'i_age_days',
 'is_cat',
 'sex_female',
 'sex_unknown',
 'status_fixed',
 'no_name',
 'cond_injured',
 'cond_neonatal',
 'cond_nursing',
 'cond_other',
 'cond_sick',
 'i_type_abandoned',
 'i_type_other',
 'i_type_owner_surrender',
 'i_type_public_assist',
 'c_breed_domestic_longhair',
 'c_breed_domestic_longhair_mix',
 'c_breed_domestic_medium_hair',
 'c_breed_domestic_medium_hair_mix',
 'c_breed_domestic_shorthair',
 'c_breed_other',
 'c_breed_siamese_mix',
 'c_color_black',
 'c_color_black_white',
 'c_color_blue',
 'c_color_blue_tabby',
 'c_color_blue_tabby_white',
 'c_color_blue_white',
 'c_color_brown_tabby_white',
 'c_color_calico',
 'c_color_cream_tabby',
 'c_color_lynx_point',
 'c_color_orange_tabby',
 'c_color_orange_tabby_white',
 'c_color_other',
 'c_color_torbie',
 'c_color_tortie',
 'c_color_white_black',
 'c_color_white_brown_tabby']

------------

## **Dogs**

### Breeds

In [29]:
# look at top 20 dog breeds (%) 
print((intake[intake['is_cat'] == 0]['Breed'].value_counts(normalize=True)*100).head(20))


Breed
Pit Bull Mix                 10.667174
Labrador Retriever Mix        9.309995
Chihuahua Shorthair Mix       7.281625
German Shepherd Mix           4.285050
Pit Bull                      3.688906
Labrador Retriever            2.160494
Chihuahua Shorthair           2.128784
Australian Cattle Dog Mix     2.094960
German Shepherd               1.950152
Boxer Mix                     1.265221
Dachshund Mix                 1.253594
Border Collie Mix             1.214485
Siberian Husky Mix            1.036910
Miniature Poodle Mix          0.978776
Australian Shepherd Mix       0.926983
Siberian Husky                0.924869
Catahoula Mix                 0.902672
Staffordshire Mix             0.843480
Great Pyrenees Mix            0.826569
Rat Terrier Mix               0.772662
Name: proportion, dtype: float64


In [30]:
print('\nNumber of unique breeds: ', intake[intake['is_cat'] == 0]['Breed'].nunique())


Number of unique breeds:  2655


In [31]:
    
# apply mixed breed name cleaning to dogs
# (convert mixed types such as 'Pit Bull/Labrador Retriever' to 'Pit Bull Mix')
intake.loc[intake['is_cat'] == 0, 'Breed'] = intake.loc[intake['is_cat'] == 0, 'Breed'].apply(clean_mixes)

# check [cleaned] top breeds and %
print((intake[intake['is_cat'] == 0]['Breed'].value_counts(normalize=True)*100).head(25))


Breed
Labrador Retriever Mix       12.151192
Pit Bull Mix                 11.653349
Chihuahua Shorthair Mix       8.533105
German Shepherd Mix           5.467825
Pit Bull                      3.688906
Australian Cattle Dog Mix     2.712244
Labrador Retriever            2.160494
Chihuahua Shorthair           2.128784
German Shepherd               1.950152
Dachshund Mix                 1.906815
Boxer Mix                     1.680619
Border Collie Mix             1.674277
Siberian Husky Mix            1.333925
Miniature Poodle Mix          1.212371
Catahoula Mix                 1.181718
Australian Shepherd Mix       1.175376
Great Pyrenees Mix            1.142610
Beagle Mix                    1.100330
Jack Russell Terrier Mix      0.974548
Pointer Mix                   0.967149
Yorkshire Terrier Mix         0.963978
Rat Terrier Mix               0.956579
Miniature Schnauzer Mix       0.944952
Siberian Husky                0.924869
Staffordshire Mix             0.917470
Name: proportion, d

In [32]:
print('\nNumber of unique breeds after cleaning: ', intake[intake['is_cat'] == 0]['Breed'].nunique())


Number of unique breeds after cleaning:  392


In [33]:

# list of significant (>=1%) dog breeds
top_dog_breeds = ['Labrador Retriever Mix', 'Pit Bull Mix', 'Chihuahua Shorthair Mix', 'German Shepherd Mix', 'Pit Bull', 'Australian Cattle Dog Mix',
                  'Labrador Retriever', 'Chihuahua Shorthair', 'German Shepherd', 'Dachshund Mix', 'Boxer Mix', 'Border Collie Mix', 'Siberian Husky Mix',
                  'Miniature Poodle Mix', 'Catahoula Mix', 'Australian Shepherd Mix', 'Great Pyrenees Mix', 'Beagle Mix']

def check_dog_breed(dog):
    if dog['is_cat'] == 0:                       # if dog:
        if dog['Breed'] in top_dog_breeds:       # if breed is in list:
            return dog['Breed'].lower()          # keep the name (lowercase)
        else:
            return 'other'       # not in list: go to 'other'
    return None                  # not a dog: skip it.


# apply the function to create a temp column
intake['Temp_dogBreed'] = intake.apply(check_dog_breed, axis=1)

# clean the breed string for use as dummy columns (replacing ' ' with '_')
intake['Temp_dogBreed'] = intake['Temp_dogBreed'].str.replace(' ', '_')

# create the dummy columns
dummies = pd.get_dummies(intake['Temp_dogBreed'], prefix='d_breed', dtype=int)

# add dummy columns to table
intake = pd.concat([intake, dummies], axis=1)

# drop temp column and baseline breed ('Labrador Retriever Mix')
intake = intake.drop(columns= ['Temp_dogBreed', 'd_breed_labrador_retriever_mix'])


# check that the column names are correctly updated
intake.columns.tolist()


['Animal ID',
 'Breed',
 'Color',
 'i_datetime',
 'i_month',
 'i_year',
 'i_age_days',
 'is_cat',
 'sex_female',
 'sex_unknown',
 'status_fixed',
 'no_name',
 'cond_injured',
 'cond_neonatal',
 'cond_nursing',
 'cond_other',
 'cond_sick',
 'i_type_abandoned',
 'i_type_other',
 'i_type_owner_surrender',
 'i_type_public_assist',
 'c_breed_domestic_longhair',
 'c_breed_domestic_longhair_mix',
 'c_breed_domestic_medium_hair',
 'c_breed_domestic_medium_hair_mix',
 'c_breed_domestic_shorthair',
 'c_breed_other',
 'c_breed_siamese_mix',
 'c_color_black',
 'c_color_black_white',
 'c_color_blue',
 'c_color_blue_tabby',
 'c_color_blue_tabby_white',
 'c_color_blue_white',
 'c_color_brown_tabby_white',
 'c_color_calico',
 'c_color_cream_tabby',
 'c_color_lynx_point',
 'c_color_orange_tabby',
 'c_color_orange_tabby_white',
 'c_color_other',
 'c_color_torbie',
 'c_color_tortie',
 'c_color_white_black',
 'c_color_white_brown_tabby',
 'd_breed_australian_cattle_dog_mix',
 'd_breed_australian_shepherd_

### Colors

In [34]:
# look at top 20 dog colors (%)
print((intake[intake['is_cat'] == 0]['Color'].value_counts(normalize=True)*100).head(25))

Color
Black/White            11.841493
Brown/White             5.686623
White                   5.497421
Tan/White               5.219432
Black                   5.215204
Tan                     4.342128
Brown                   4.088449
Black/Tan               3.832657
Tricolor                3.655082
Black/Brown             3.637113
White/Black             3.627600
White/Brown             3.272451
Blue/White              3.125528
Brown Brindle/White     3.008202
Brown/Black             2.971208
White/Tan               2.511416
Red/White               1.749324
Tan/Black               1.713386
Red                     1.682733
Brown Brindle           1.559065
Chocolate/White         1.229283
Cream                   0.958693
Yellow                  0.802258
Fawn/White              0.785346
Sable                   0.736724
Name: proportion, dtype: float64


In [35]:
# list of significant (>=1%) dog colors
top_dog_colors = ['Black/White', 'Brown/White', 'White', 'Tan/White', 'Black', 'Tan', 'Brown', 'Black/Tan', 'Tricolor', 'Black/Brown', 'White/Black',
                  'White/Brown', 'Blue/White', 'Brown Brindle/White', 'Brown/Black', 'White/Tan', 'Red/White', 'Tan/Black', 'Red', 'Brown Brindle',
                  'Chocolate/White']

def check_dog_color(dog):
    if dog['is_cat'] == 0:                       # if a dog:
        if dog['Color'] in top_dog_colors:       # if color is in list:
            return dog['Color'].lower()          # keep the name (lowercase)
        else:
            return 'other'       # not in list: goes to 'other'
    return None                  # not a dog: skip it


# apply the function to create a temp column
intake['Temp_dogColor'] = intake.apply(check_dog_color, axis=1)

# clean the color string for use as dummy columns (replacing ' ' and '/' with '_')
intake['Temp_dogColor'] = intake['Temp_dogColor'].str.replace(' ', '_').str.replace('/', '_')

# create the dummy columns
dummies = pd.get_dummies(intake['Temp_dogColor'], prefix='d_color', dtype=int)

# add dummy columns to table
intake = pd.concat([intake, dummies], axis=1)

# drop temp column and baseline ('Black/White')
intake = intake.drop(columns=['Temp_dogColor', 'd_color_black_white'])


# check that the column names are correctly updated
intake.columns.tolist()


['Animal ID',
 'Breed',
 'Color',
 'i_datetime',
 'i_month',
 'i_year',
 'i_age_days',
 'is_cat',
 'sex_female',
 'sex_unknown',
 'status_fixed',
 'no_name',
 'cond_injured',
 'cond_neonatal',
 'cond_nursing',
 'cond_other',
 'cond_sick',
 'i_type_abandoned',
 'i_type_other',
 'i_type_owner_surrender',
 'i_type_public_assist',
 'c_breed_domestic_longhair',
 'c_breed_domestic_longhair_mix',
 'c_breed_domestic_medium_hair',
 'c_breed_domestic_medium_hair_mix',
 'c_breed_domestic_shorthair',
 'c_breed_other',
 'c_breed_siamese_mix',
 'c_color_black',
 'c_color_black_white',
 'c_color_blue',
 'c_color_blue_tabby',
 'c_color_blue_tabby_white',
 'c_color_blue_white',
 'c_color_brown_tabby_white',
 'c_color_calico',
 'c_color_cream_tabby',
 'c_color_lynx_point',
 'c_color_orange_tabby',
 'c_color_orange_tabby_white',
 'c_color_other',
 'c_color_torbie',
 'c_color_tortie',
 'c_color_white_black',
 'c_color_white_brown_tabby',
 'd_breed_australian_cattle_dog_mix',
 'd_breed_australian_shepherd_

-----
## ***Post-Encoding Feature Cleanup***

In [36]:
# Drop the original categorical columns for both species
intake = intake.drop(columns=['Breed', 'Color'])

In [37]:

# look at intake table columns and data types
print(intake.info())

<class 'pandas.core.frame.DataFrame'>
Index: 163932 entries, 0 to 173811
Data columns (total 82 columns):
 #   Column                             Non-Null Count   Dtype         
---  ------                             --------------   -----         
 0   Animal ID                          163932 non-null  object        
 1   i_datetime                         163932 non-null  datetime64[ns]
 2   i_month                            163932 non-null  int32         
 3   i_year                             163932 non-null  int32         
 4   i_age_days                         163932 non-null  int64         
 5   is_cat                             163932 non-null  int64         
 6   sex_female                         163932 non-null  int64         
 7   sex_unknown                        163932 non-null  int64         
 8   status_fixed                       163932 non-null  int64         
 9   no_name                            163932 non-null  int64         
 10  cond_injured             

-----
# **Merge and Final Clean**

### Extract Necessary Features from Outcome Table

In [38]:
# o_datetime: to calculate the Length of Stay
# Animal ID: for matching rows on merge

outcome_final = outcome[['Animal ID', 'o_datetime']].copy()  # copy these features to new variable

# check new df's features and sample of values
outcome_final


,Animal ID,o_datetime
12,A683090,2014-07-11 05:00:00
42,A693369,2015-03-11 05:00:00
43,A698568,2015-03-14 05:00:00
62,A699357,2015-04-02 05:00:00
63,A699358,2015-04-03 05:00:00
...,...,...
173755,A928232,2025-05-04 23:08:00
173756,A464421,2025-05-04 23:23:00
173757,A925997,2025-05-04 23:39:00
173758,A929205,2025-05-04 23:46:00


### Merge Tables

In [39]:

# merge the tables with inner join on Animal ID, creating new 'master' df
# (dropping any records from intake df whose 'Animal ID' doesn't match a record in outcome_final df (previously narrowed to adoption only outcomes) )
master = pd.merge(intake, outcome_final, on='Animal ID', how='inner')
print(master.shape)


(107588, 83)


### Calculate Length of Stay

In [40]:

# calculate the Length of Stay ( (outcome date/time) - (income date/time) )     
master['length_of_stay'] = (master['o_datetime'] - master['i_datetime']).dt.total_seconds() / 86400    # (total seconds of stay/seconds per day)


### Handle Erroneous Duplicate Stays

In [41]:

# print number of records in master df after merge
print(f'  # of records at merge: {len(master):,}')

# print number of records with negative stays (created on merge by animal IDs with multiple stays)
print(f'- # of "time travelers": {(master['length_of_stay'] < 0).sum():,}')

# remove "time travelers" (an outcome occuring prior to intake)
master = master[master['length_of_stay'] >= 0]

print('----------------------------------------')
print(f'# of records with positive stays: {len(master):,}')
print()

# for animals with multiple stays, sort their stays by length in ascending order:
# sort first by animalID, then sort by length of stay 
# (ensures intake is matched with its *true* next outcome, not an outcome years later)
master = master.sort_values(by=['Animal ID', 'length_of_stay'])

# drop duplicates:
# now that LOS's are correctly sorted,
# if an Animal ID appears twice on the exact same Date/Time, keep the FIRST one 
master = master.drop_duplicates(subset=['Animal ID', 'i_datetime'], keep='first')

# check the record count after removing erroneous duplicates 
print(f'# of records after removing erroneous duplicates: {len(master):,}')


  # of records at merge: 107,588
- # of "time travelers": 12,997
----------------------------------------
# of records with positive stays: 94,591

# of records after removing erroneous duplicates: 84,805


## Add Categorical Bins & Separate Attributes from Target

In [42]:

# define the boundaries
# 0-6.9 days = nominal
# 7-30.9 days = elevated
# 31-89.9 days = severe
# 90+ days     = critical
bins = [0, 7, 31, 90, float('inf')]
labels = ['Nominal', 'Elevated', 'Severe', 'Critical']

# create the categorical column
master['long_stay_risk'] = pd.cut(
    master['length_of_stay'],
    bins=bins, 
    labels=labels, 
    right=False # include the left number, exclude the right
) 

# drop the animal_id column and datetime columns
master = master.drop(columns=['length_of_stay', 'Animal ID', 'i_datetime', 'o_datetime'])

# _____________________________________________________


from sklearn.model_selection  import train_test_split

# separate attributes (x) from target (y)
attributes = master.drop(columns=['long_stay_risk'])
target = master['long_stay_risk']

# final check
print('----------------------=== FINAL SHAPES ===-----------------------\n')
print('MASTER')
print(master.shape)
print()
print('ATTRIBUTES (x)')
print(attributes.shape)
print()
print('TARGET (y)')
print(target.shape)
print()
print('------------------=== FULL MASTER DATAFRAME ===-------------------\n')
print(master.info())
print('\n----------------------=== ATTRIBUTES ===-------------------------\n')
print(attributes.info())
print('\n------------------------=== TARGET ===---------------------------\n')
print(target)


----------------------=== FINAL SHAPES ===-----------------------

MASTER
(84805, 81)

ATTRIBUTES (x)
(84805, 80)

TARGET (y)
(84805,)

------------------=== FULL MASTER DATAFRAME ===-------------------

<class 'pandas.core.frame.DataFrame'>
Index: 84805 entries, 66 to 107579
Data columns (total 81 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   i_month                            84805 non-null  int32   
 1   i_year                             84805 non-null  int32   
 2   i_age_days                         84805 non-null  int64   
 3   is_cat                             84805 non-null  int64   
 4   sex_female                         84805 non-null  int64   
 5   sex_unknown                        84805 non-null  int64   
 6   status_fixed                       84805 non-null  int64   
 7   no_name                            84805 non-null  int64   
 8   cond_injured                       

In [43]:

print(target.value_counts(normalize=True)*100) # view frequency of each target risk tiers (%)

long_stay_risk
Elevated    33.957903
Nominal     30.547727
Severe      25.523259
Critical     9.971110
Name: proportion, dtype: float64


In [44]:

print('Missing values in attributes (x): ', attributes.isna().sum().sum()) # check that the features table has no missing values before testing

Missing values in attributes (x):  0


-----
# **Model Testing**

## Data Split & Category Weight

In [45]:
from sklearn.model_selection import train_test_split

# split train/test 80/20
x_train_full, x_test, y_train_full, y_test = train_test_split(
    attributes, 
    target, 
    test_size=0.2, 
    stratify=target, 
    random_state=321
)
# validation split
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, 
    test_size=0.10, 
    stratify=y_train_full, 
    random_state=321
)

# define weights so that the 'Critical' category penalizes more for a mis-classified animal
custom_weights = {'Nominal': 1.0, 'Elevated': 1.0, 'Severe': 1.5, 'Critical': 5.0}

## **Logistic Regression**

In [46]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import *
from sklearn import metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import *
from sklearn.model_selection import KFold

import numpy as np

# stratified cross validation with 5 folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=321)

fold = 0
f1 = []
precision = []
recall = []
accuracy = []
features = []

for train_index, val_index in skf.split(x_train_full, y_train_full):
    fold += 1
    print ("Fold %d" % fold)
    # partition: training/validation
    train_x, val_x = x_train_full.iloc[train_index], x_train_full.iloc[val_index]
    train_y, val_y = y_train_full.iloc[train_index], y_train_full.iloc[val_index]

    # feature selection
    f_val, p_val = chi2(train_x, train_y)

    # print chi2 and p values       
    df_scores = pd.DataFrame(zip(train_x.columns, f_val, p_val), columns=["feature", "chi2", "p"])
    df_scores["chi2"] = df_scores["chi2"].round(2)
    df_scores["p"] = df_scores["p"].round(3)
    
    # use significant features (p < 0.05) 
    sel_features = df_scores[df_scores["p"]<0.05]["feature"].values
    
    #scaling
    scaler = StandardScaler()
    train_x_scaled = pd.DataFrame(scaler.fit_transform(train_x[sel_features]), columns=sel_features)
    val_x_scaled = pd.DataFrame(scaler.transform(val_x[sel_features]), columns=sel_features)
 
    # train model
    clf = LogisticRegression(random_state=fold, class_weight = custom_weights, solver='liblinear')
    # hyperparameter tuning parameters
    grid_values = {'penalty': ['l1', 'l2'], 'C':[0.001,0.01,1,5,10,25]}

    # grid search
    grid_clf_acc = GridSearchCV(clf, param_grid = grid_values, scoring = 'accuracy')
    grid_clf_acc.fit(train_x_scaled, train_y)

    # predict
    pred = grid_clf_acc.predict(val_x_scaled)

    # classification results
    for line in metrics.classification_report(val_y, pred).split("\n"):
        print (line)
        
    f1.append(metrics.f1_score(val_y, pred, average='micro'))
    precision.append(metrics.precision_score(val_y, pred, average='micro'))
    recall.append(metrics.recall_score(val_y, pred, average='micro'))
    accuracy.append(metrics.accuracy_score(val_y, pred))
    print("Best parameters for Fold %d: %s" % (fold, grid_clf_acc.best_params_))
    print()

print ("Average F1: %.2f" % np.mean(f1))
print ("Average prcesion: %.2f" % np.mean(precision))
print ("Average recall: %.2f" % np.mean(recall))
print ("Average accuracy: %.2f" % np.mean(accuracy))

Fold 1
              precision    recall  f1-score   support

    Critical       0.22      0.45      0.29      1353
    Elevated       0.43      0.21      0.28      4608
     Nominal       0.53      0.57      0.55      4145
      Severe       0.43      0.51      0.47      3463

    accuracy                           0.42     13569
   macro avg       0.40      0.43      0.40     13569
weighted avg       0.44      0.42      0.41     13569

Best parameters for Fold 1: {'C': 5, 'penalty': 'l2'}

Fold 2
              precision    recall  f1-score   support

    Critical       0.22      0.47      0.30      1353
    Elevated       0.45      0.20      0.28      4608
     Nominal       0.53      0.58      0.56      4145
      Severe       0.43      0.49      0.46      3463

    accuracy                           0.42     13569
   macro avg       0.41      0.44      0.40     13569
weighted avg       0.45      0.42      0.41     13569

Best parameters for Fold 2: {'C': 10, 'penalty': 'l1'}

Fold 

## **Decision Tree**

In [47]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import *
from sklearn import metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier # Decision Tree
from sklearn.feature_selection import *
from sklearn.model_selection import KFold

import numpy as np

# stratified cross validation with 5 folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=321)

fold = 0
f1 = []
precision = []
recall = []
accuracy = []
features = []

for train_index, val_index in skf.split(x_train_full, y_train_full):
    fold += 1
    print ("Fold %d" % fold)
    
    # partition: training/validation
    train_x, val_x = x_train_full.iloc[train_index], x_train_full.iloc[val_index]
    train_y, val_y = y_train_full.iloc[train_index], y_train_full.iloc[val_index]

    # feature selection
    f_val, p_val = chi2(train_x, train_y)

    # chi2 and p values       
    df_scores = pd.DataFrame(zip(train_x.columns, f_val, p_val), columns=["feature", "chi2", "p"])
    df_scores["chi2"] = df_scores["chi2"].round(2)
    df_scores["p"] = df_scores["p"].round(3)
    
    # use significant features (p < 0.05) 
    sel_features = df_scores[df_scores["p"]<0.05]["feature"].values
    
    #scaling
    scaler = StandardScaler()
    train_x_scaled = pd.DataFrame(scaler.fit_transform(train_x[sel_features]), columns=sel_features)
    val_x_scaled = pd.DataFrame(scaler.transform(val_x[sel_features]), columns=sel_features)
 
    # train model
    clf = DecisionTreeClassifier(random_state=fold, class_weight = custom_weights)
    # hyperparameter tuning parameters
    grid_values = {'criterion': ['gini', 'entropy'], 'max_depth':[None, 5, 10, 25, 50]}

    # grid search
    grid_clf_acc = GridSearchCV(clf, param_grid = grid_values, scoring = 'accuracy')
    grid_clf_acc.fit(train_x_scaled, train_y)

    # predict
    pred = grid_clf_acc.predict(val_x_scaled)

    # classification results
    for line in metrics.classification_report(val_y, pred).split("\n"):
        print (line)
        
    f1.append(metrics.f1_score(val_y, pred, average='micro'))
    precision.append(metrics.precision_score(val_y, pred, average='micro'))
    recall.append(metrics.recall_score(val_y, pred, average='micro'))
    accuracy.append(metrics.accuracy_score(val_y, pred))
    print("Best parameters for Fold %d: %s" % (fold, grid_clf_acc.best_params_))
    print()

print ("Average F1: %.2f" % np.mean(f1))
print ("Average prcesion: %.2f" % np.mean(precision))
print ("Average recall: %.2f" % np.mean(recall))
print ("Average accuracy: %.2f" % np.mean(accuracy))

Fold 1
              precision    recall  f1-score   support

    Critical       0.18      0.46      0.26      1353
    Elevated       0.44      0.32      0.37      4608
     Nominal       0.59      0.50      0.54      4145
      Severe       0.52      0.47      0.50      3463

    accuracy                           0.43     13569
   macro avg       0.43      0.44      0.42     13569
weighted avg       0.48      0.43      0.45     13569

Best parameters for Fold 1: {'criterion': 'gini', 'max_depth': 25}

Fold 2
              precision    recall  f1-score   support

    Critical       0.20      0.31      0.24      1353
    Elevated       0.43      0.39      0.41      4608
     Nominal       0.57      0.48      0.52      4145
      Severe       0.47      0.51      0.49      3463

    accuracy                           0.44     13569
   macro avg       0.42      0.42      0.41     13569
weighted avg       0.46      0.44      0.45     13569

Best parameters for Fold 2: {'criterion': 'entro

## **Random Forest**

In [48]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import *
from sklearn import metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier # Decision Tree
from sklearn.feature_selection import *
from sklearn.model_selection import KFold

import numpy as np

# stratified cross validation with 5 folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=321)

fold = 0
f1 = []
precision = []
recall = []
accuracy = []
features = []

for train_index, val_index in skf.split(x_train_full, y_train_full):
    fold += 1
    print ("Fold %d" % fold)
    
    # partition: training/validation
    train_x, val_x = x_train_full.iloc[train_index], x_train_full.iloc[val_index]
    train_y, val_y = y_train_full.iloc[train_index], y_train_full.iloc[val_index]

    # feature selection
    f_val, p_val = chi2(train_x, train_y)

    # print chi2 and p values       
    df_scores = pd.DataFrame(zip(train_x.columns, f_val, p_val), columns=["feature", "chi2", "p"])
    df_scores["chi2"] = df_scores["chi2"].round(2)
    df_scores["p"] = df_scores["p"].round(3)
    
    # use significant features (p < 0.05) 
    sel_features = df_scores[df_scores["p"]<0.05]["feature"].values
    
    #scaling
    scaler = StandardScaler()
    train_x_scaled = pd.DataFrame(scaler.fit_transform(train_x[sel_features]), columns=sel_features)
    val_x_scaled = pd.DataFrame(scaler.transform(val_x[sel_features]), columns=sel_features)
 
    # train model
    clf = RandomForestClassifier(random_state=fold, class_weight = custom_weights)
    # hyperparameter tuning parameters
    grid_values = {'n_estimators': [50, 100], 'max_features':['sqrt', 'log2'], 'max_depth': [None, 10, 20]}

    # grid search
    grid_clf_acc = GridSearchCV(clf, param_grid = grid_values, scoring = 'accuracy')
    grid_clf_acc.fit(train_x_scaled, train_y)

    # predict
    pred = grid_clf_acc.predict(val_x_scaled)

    # classification results
    for line in metrics.classification_report(val_y, pred).split("\n"):
        print (line)
        
    f1.append(metrics.f1_score(val_y, pred, average='micro'))
    precision.append(metrics.precision_score(val_y, pred, average='micro'))
    recall.append(metrics.recall_score(val_y, pred, average='micro'))
    accuracy.append(metrics.accuracy_score(val_y, pred))
    print("Best parameters for Fold %d: %s" % (fold, grid_clf_acc.best_params_))
    print()

print ("Average F1: %.2f" % np.mean(f1))
print ("Average prcesion: %.2f" % np.mean(precision))
print ("Average recall: %.2f" % np.mean(recall))
print ("Average accuracy: %.2f" % np.mean(accuracy))

Fold 1
              precision    recall  f1-score   support

    Critical       0.21      0.24      0.22      1353
    Elevated       0.43      0.42      0.43      4608
     Nominal       0.57      0.56      0.56      4145
      Severe       0.52      0.52      0.52      3463

    accuracy                           0.47     13569
   macro avg       0.43      0.43      0.43     13569
weighted avg       0.47      0.47      0.47     13569

Best parameters for Fold 1: {'max_depth': None, 'max_features': 'sqrt', 'n_estimators': 100}

Fold 2
              precision    recall  f1-score   support

    Critical       0.23      0.26      0.24      1353
    Elevated       0.44      0.42      0.43      4608
     Nominal       0.57      0.56      0.57      4145
      Severe       0.51      0.52      0.51      3463

    accuracy                           0.47     13569
   macro avg       0.44      0.44      0.44     13569
weighted avg       0.47      0.47      0.47     13569

Best parameters for Fo

## **Historical Gradient Boosting Classifier**
##### *Because the 'tree' models gave better baseline results, I'm going to try a boosted model*

In [49]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt

fold = 0
f1 = []
precision = []
recall = []
accuracy = []
features = []

for train_index, val_index in skf.split(x_train_full, y_train_full):
    fold += 1
    print ("Fold %d" % fold)
    
    # partition: training/validation
    train_x, val_x = x_train_full.iloc[train_index], x_train_full.iloc[val_index]
    train_y, val_y = y_train_full.iloc[train_index], y_train_full.iloc[val_index]

    # feature selection
    f_val, p_val = chi2(train_x, train_y)

    # print chi2 and p values       
    df_scores = pd.DataFrame(zip(train_x.columns, f_val, p_val), columns=["feature", "chi2", "p"])
    df_scores["chi2"] = df_scores["chi2"].round(2)
    df_scores["p"] = df_scores["p"].round(3)
    
    # use significant features (p < 0.05) 
    sel_features = df_scores[df_scores["p"]<0.05]["feature"].values
    
    #scaling
    scaler = StandardScaler()
    train_x_scaled = pd.DataFrame(scaler.fit_transform(train_x[sel_features]), columns=sel_features)
    val_x_scaled = pd.DataFrame(scaler.transform(val_x[sel_features]), columns=sel_features)

    # create custom weight map
    c_weights = compute_sample_weight(class_weight = custom_weights, y=train_y)
    
    # train model
    clf = HistGradientBoostingClassifier(random_state=fold)
    # hyperparameter tuning parameters
    grid_values = {'learning_rate': [0.01, 0.1], 'max_iter':[100, 200], 'max_depth': [5, 10]}

    # grid search
    grid_clf_acc = GridSearchCV(clf, param_grid = grid_values, scoring = 'accuracy')
    grid_clf_acc.fit(train_x_scaled, train_y, sample_weight = c_weights)

    # predict
    pred = grid_clf_acc.predict(val_x_scaled)

    # classification results
    for line in metrics.classification_report(val_y, pred).split("\n"):
        print (line)
        
    f1.append(metrics.f1_score(val_y, pred, average='micro'))
    precision.append(metrics.precision_score(val_y, pred, average='micro'))
    recall.append(metrics.recall_score(val_y, pred, average='micro'))
    accuracy.append(metrics.accuracy_score(val_y, pred))
    print("Best parameters for Fold %d: %s" % (fold, grid_clf_acc.best_params_))
    print()
    
print ("Average F1: %.2f" % np.mean(f1))
print ("Average prcesion: %.2f" % np.mean(precision))
print ("Average recall: %.2f" % np.mean(recall))
print ("Average accuracy: %.2f" % np.mean(accuracy))

Fold 1
              precision    recall  f1-score   support

    Critical       0.19      0.64      0.30      1353
    Elevated       0.48      0.28      0.35      4608
     Nominal       0.65      0.58      0.61      4145
      Severe       0.61      0.48      0.54      3463

    accuracy                           0.46     13569
   macro avg       0.48      0.49      0.45     13569
weighted avg       0.54      0.46      0.47     13569

Best parameters for Fold 1: {'learning_rate': 0.1, 'max_depth': 10, 'max_iter': 100}

Fold 2
              precision    recall  f1-score   support

    Critical       0.19      0.64      0.29      1353
    Elevated       0.51      0.28      0.36      4608
     Nominal       0.64      0.57      0.60      4145
      Severe       0.61      0.48      0.53      3463

    accuracy                           0.46     13569
   macro avg       0.49      0.49      0.45     13569
weighted avg       0.54      0.46      0.47     13569

Best parameters for Fold 2: {'

-----
# **Final Test (HGBC)**

In [51]:
from sklearn.ensemble import HistGradientBoostingClassifier

# calculate chi2 on fulll training set
f_val, p_val = chi2(x_train_full, y_train_full)

df_scores = pd.DataFrame(zip(x_train_full.columns, f_val, p_val), columns=["feature", "chi2", "p"])
df_scores["chi2"] = df_scores["chi2"].round(2)
df_scores["p"] = df_scores["p"].round(3)
    
# use significant features (p < 0.05) 
sel_features = df_scores[df_scores["p"]<0.05]["feature"].values

# scale
scaler = StandardScaler()
x_train_full_scaled = pd.DataFrame(scaler.fit_transform(x_train_full[sel_features]), columns = sel_features)
x_test_scaled = pd.DataFrame(scaler.transform(x_test[sel_features]), columns = sel_features)

# train with best hgbc parameters 
final_hgbc = HistGradientBoostingClassifier(
    random_state=321,
    learning_rate=0.1,
    max_depth=10,
    max_iter=100
)

# calculate weights for full training set
final_c_weights = compute_sample_weight(class_weight=custom_weights, y=y_train_full)

# train on full training set with custom weights
final_hgbc.fit(x_train_full_scaled, y_train_full, sample_weight = final_c_weights)

# predict
train_pred = final_hgbc.predict(x_train_full_scaled)
test_pred = final_hgbc.predict(x_test_scaled)

# accuracy and gap 
train_acc = accuracy_score(y_train_full, train_pred)
test_acc = accuracy_score(y_test, test_pred)
gap = train_acc - test_acc

# print:
# metrics
print("f1: " + str(f1_score(y_test, test_pred, average = 'micro'))) 
print("accuracy: " + str(accuracy_score(y_test, test_pred)))
print("precision: " + str(precision_score(y_test, test_pred, average = 'micro')))
print("recall: " + str(recall_score(y_test, test_pred, average = 'micro')))
print()
print("-" * 30) 
# accuracy: training/testing
print(f"\ntraining accuracy: {train_acc:.4f}")
print(f"test accuracy: {test_acc:.4f}")
print(f"gap: {gap:.4f}")
print()
print("-" * 30)
# final report
print("\nClassification Report:\n")
print(metrics.classification_report(y_test, test_pred))

f1: 0.46229585519721716
accuracy: 0.46229585519721716
precision: 0.46229585519721716
recall: 0.46229585519721716

------------------------------

training accuracy: 0.4852
test accuracy: 0.4623
gap: 0.0229

------------------------------

Classification Report:

              precision    recall  f1-score   support

    Critical       0.20      0.67      0.30      1691
    Elevated       0.50      0.29      0.36      5760
     Nominal       0.64      0.58      0.61      5181
      Severe       0.64      0.47      0.54      4329

    accuracy                           0.46     16961
   macro avg       0.49      0.50      0.46     16961
weighted avg       0.55      0.46      0.48     16961



# **Artifical Neural Network**

In [52]:
# pip install tensorflow

In [53]:
from keras.models import Sequential
from keras import layers
import numpy as np
from sklearn import metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

input_dim = x_train_full_scaled.shape[1] # number of features

# custom weighting for Keras categories
custom_weights_k = {0: 1.0, 1: 1.0, 2: 1.5, 3: 5.0}

model = Sequential()
model.add(layers.Dense(10, input_dim = input_dim, activation='relu'))

# 4 nodes for 4 categories, softmax for multiclass
model.add(layers.Dense(4, activation='softmax'))

# convert category datatypes to integers for keras
y_train_full = y_train_full.cat.codes
y_test = y_test.cat.codes

model.compile(loss='sparse_categorical_crossentropy',
              optimizer = 'adam',
              metrics=['accuracy'])

# train the model
history = model.fit(x_train_full_scaled, y_train_full,
                    epochs=100,
                    verbose=False,
                    validation_data=(x_test_scaled, y_test),
                    batch_size=10,
                    class_weight = custom_weights_k)

# print model evaluation
loss, accuracy = model.evaluate(x_train_full_scaled, y_train_full, verbose=False)
print("Training Accuracy: {:.4f}".format(accuracy))
loss, accuracy = model.evaluate(x_test_scaled, y_test, verbose=False)
print("Testing Accuracy: {:.4f}\n".format(accuracy))

# predict/generate metrics
pred_probs = model.predict(x_test_scaled)
pred_y = np.argmax(pred_probs, axis = 1)

print("f1: " + str(f1_score(y_test, pred_y, average = 'micro'))) 
print("accuracy: " + str(accuracy_score(y_test, pred_y)))
print("precision: " + str(precision_score(y_test, pred_y, average = 'micro')))
print("recall: " + str(recall_score(y_test, pred_y, average = 'micro')))
print("\n", ("-" * 30)) # border
# final report
print("\nClassification Report:\n")
print(metrics.classification_report(y_test, pred_y))

Training Accuracy: 0.4418
Testing Accuracy: 0.4358

531/531 ━━━━━━━━━━━━━━━━━━━━ 0s 690us/step
f1: 0.4358233594717293
accuracy: 0.4358233594717293
precision: 0.4358233594717293
recall: 0.4358233594717293

 ------------------------------

Classification Report:

              precision    recall  f1-score   support

           0       0.61      0.55      0.58      5181
           1       0.48      0.25      0.33      5760
           2       0.57      0.47      0.52      4329
           3       0.19      0.64      0.29      1691

    accuracy                           0.44     16961
   macro avg       0.46      0.48      0.43     16961
weighted avg       0.51      0.44      0.45     16961

